In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
date_rng = pd.date_range(start='2019-08-01 00:00:00', end='2019-08-31 23:00:00', freq='H')

In [ ]:
ori_ghi = pd.read_csv('../dataset/dataset_fed/ghi/ghi_dataset.csv')

In [ ]:
ghi_df = pd.read_csv('../result_fedlstm/ghi/pred_vs_true.csv')
temp_df = pd.read_csv('../result_fedlstm/temperature/pred_vs_true.csv')
kw_df = pd.read_csv('../result_fedlstm/kw/pred_vs_true.csv')

In [ ]:
ori_ghi.head()

In [ ]:
new_ghi = []
for i in range(len(ori_ghi)):
    for j in range(len(ghi_df)):
        if ori_ghi['KW'].iloc[i] == ghi_df['true'].iloc[j]:
            result = {
                'date': ori_ghi['date'].iloc[i],
                'time': ori_ghi['time'].iloc[i],
                'pred': ghi_df['pred'].iloc[j + 2],
                'true': ghi_df['true'].iloc[j + 2]
            }
            new_ghi.append(result)

new_ghi_df = pd.DataFrame(new_ghi)

In [ ]:
new_ghi_df.head()

In [ ]:
new_ghi_df.tail()

In [ ]:
ori_temp = pd.read_csv('../dataset/dataset_fed/temperature/temperature_dataset.csv')

In [ ]:
new_temperature = []
for i in range(len(ori_temp)):
    for j in range(len(temp_df)):
        if ori_temp['KW'].iloc[i] == temp_df['true'].iloc[j]:
            result = {
                'date': ori_temp['date'].iloc[i],
                'time': ori_temp['time'].iloc[i],
                'pred': temp_df['pred'].iloc[j + 2],
                'true': temp_df['true'].iloc[j + 2]
            }
            new_temperature.append(result)

new_temp_df = pd.DataFrame(new_temperature)

In [ ]:
ori_kw = pd.read_csv('../dataset/dataset_fed/kw/kw_dataset.csv')

In [ ]:
new_kw = []
for i in range(len(ori_kw)):
    for j in range(len(kw_df)):
        if ori_kw['OT'].iloc[i] == kw_df['true'].iloc[j]:
            result = {
                'date': ori_kw['date'].iloc[i],
                'time': ori_kw['time'].iloc[i],
                'pred': kw_df['pred'].iloc[j],
                'true': kw_df['true'].iloc[j]
            }
            new_kw.append(result)

new_kw_df = pd.DataFrame(new_kw)

In [ ]:
new_ghi_df['datetime'] = pd.to_datetime(new_ghi_df['date'] + ' ' + new_ghi_df['time'])
new_ghi_df_sorted = new_ghi_df.sort_values(by='datetime', ascending=True)

In [ ]:
new_temp_df['datetime'] = pd.to_datetime(new_temp_df['date'] + ' ' + new_temp_df['time'])
new_temp_df_sorted = new_temp_df.sort_values(by='datetime', ascending=True)

In [ ]:
new_kw_df['datetime'] = pd.to_datetime(new_kw_df['date'] + ' ' + new_kw_df['time'])
new_kw_df_sorted = new_kw_df.sort_values(by='datetime', ascending=True)

In [ ]:
new_ghi_df_sorted.head()

In [ ]:
new_kw_df_sorted.to_csv('new kw file.csv', index=False)
new_temp_df_sorted.to_csv('new temp file.csv', index=False)
new_ghi_df_sorted.to_csv('new ghi file.csv', index=False)

In [2]:
new_kw_df_sorted = pd.read_csv('new kw file.csv')
new_temp_df_sorted = pd.read_csv('new temp file.csv')
new_ghi_df_sorted = pd.read_csv('new ghi file.csv')

In [3]:
start_date = '2019-08-01 00:00:00'
end_date = '2019-08-31 23:00:00'

In [4]:
filtered_new_kw_df = new_kw_df_sorted[(new_kw_df_sorted['datetime'] >= start_date) & (new_kw_df_sorted['datetime'] <= end_date)]
filtered_new_temp_df = new_temp_df_sorted[(new_temp_df_sorted['datetime'] >= start_date) & (new_temp_df_sorted['datetime'] <= end_date)]
filtered_new_ghi_df = new_ghi_df_sorted[(new_ghi_df_sorted['datetime'] >= start_date) & (new_ghi_df_sorted['datetime'] <= end_date)]


In [5]:
filtered_new_kw_df['error'] = abs(filtered_new_kw_df['true'] - filtered_new_kw_df['pred'])
filtered_new_temp_df['error'] = abs(filtered_new_temp_df['true'] - filtered_new_temp_df['pred'])
filtered_new_ghi_df['error'] = abs(filtered_new_ghi_df['true'] - filtered_new_ghi_df['pred'])

/var/folders/97/8bx4_dmx3038pm1qrblmtrl80000gn/T/ipykernel_26926/2866521327.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_new_kw_df['error'] = abs(filtered_new_kw_df['true'] - filtered_new_kw_df['pred'])
/var/folders/97/8bx4_dmx3038pm1qrblmtrl80000gn/T/ipykernel_26926/2866521327.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_new_temp_df['error'] = abs(filtered_new_temp_df['true'] - filtered_new_temp_df['pred'])
/var/folders/97/8bx4_dmx3038pm1qrblmtrl80000gn/T/ipykernel_26926/286

In [6]:
min_error_kw_df = filtered_new_kw_df.loc[filtered_new_kw_df.groupby('datetime')['error'].idxmin()]
min_error_temp_df = filtered_new_temp_df.loc[filtered_new_temp_df.groupby('datetime')['error'].idxmin()]
min_error_ghi_df = filtered_new_ghi_df.loc[filtered_new_ghi_df.groupby('datetime')['error'].idxmin()]

In [11]:
# Gabungkan min_error_kw_df, min_error_temp_df, dan min_error_ghi_df berdasarkan 'datetime'
merged_df = pd.merge(min_error_kw_df[['datetime', 'pred']], 
                     min_error_temp_df[['datetime', 'pred']], 
                     on='datetime', 
                     how='outer', 
                     suffixes=('_kw', '_temp'))

merged_df = pd.merge(merged_df, 
                     min_error_ghi_df[['datetime', 'pred']], 
                     on='datetime', 
                     how='outer')

# Ganti nama kolom prediksi untuk lebih jelas
merged_df.columns = ['datetime', 'kw_pred', 'temp_pred', 'ghi_pred']

# Buang baris yang memiliki nilai NaN pada salah satu prediksi
merged_df = merged_df.dropna(subset=['kw_pred', 'temp_pred', 'ghi_pred'])

# Simpan ke CSV hasil gabungan
merged_df.to_csv('min_error_all_per_datetime_cleaned.csv', index=False)

# Jika ingin melihat hasilnya
print(merged_df.head())


              datetime    kw_pred  temp_pred   ghi_pred
0  2019-08-01 00:00:00  637.18660  28.778616  34.627990
1  2019-08-01 01:00:00  583.32086  28.798132  30.192660
2  2019-08-01 02:00:00  531.16560  28.781013  15.181123
3  2019-08-01 03:00:00  499.77240  28.700806  -0.248626
4  2019-08-01 04:00:00  465.63425  28.653845  -2.834060


In [12]:
# Pastikan kolom 'datetime' dalam format datetime
merged_df['datetime'] = pd.to_datetime(merged_df['datetime'])

# Ambil jam dari kolom 'datetime'
merged_df['hour'] = merged_df['datetime'].dt.hour

# Set nilai 'ghi_pred' menjadi 0 jika waktu antara 18:00:00 hingga 05:00:00
merged_df.loc[(merged_df['hour'] >= 18) | (merged_df['hour'] <= 5), 'ghi_pred'] = 0

# Hapus kolom 'hour' yang sudah tidak diperlukan
merged_df = merged_df.drop(columns=['hour'])

# Simpan hasilnya ke CSV
merged_df.to_csv('min_error_all_per_datetime_cleaned_with_ghi_zero.csv', index=False)

# Jika ingin melihat hasilnya
print(merged_df.head())

             datetime    kw_pred  temp_pred  ghi_pred
0 2019-08-01 00:00:00  637.18660  28.778616       0.0
1 2019-08-01 01:00:00  583.32086  28.798132       0.0
2 2019-08-01 02:00:00  531.16560  28.781013       0.0
3 2019-08-01 03:00:00  499.77240  28.700806       0.0
4 2019-08-01 04:00:00  465.63425  28.653845       0.0


In [13]:
# Ambil prediksi berdasarkan hasil min_error_all
pred_ghi = merged_df['ghi_pred']
pred_temp = merged_df['temp_pred']
pred_kw = merged_df['kw_pred']

# Ambil datetime yang sudah difilter
datetime_values = merged_df['datetime']

# Buat DataFrame baru dengan prediksi dan datetime
pred_all_final = pd.DataFrame({
    'datetime': datetime_values,
    'pred_ghi': pred_ghi,
    'pred_temp': pred_temp,
    'pred_kw': pred_kw
})

# Simpan ke dalam CSV
pred_all_final.to_csv('pred_all_final.csv', index=False)

# Jika ingin melihat hasilnya
print(pred_all_final)


               datetime  pred_ghi  pred_temp    pred_kw
0   2019-08-01 00:00:00       0.0  28.778616  637.18660
1   2019-08-01 01:00:00       0.0  28.798132  583.32086
2   2019-08-01 02:00:00       0.0  28.781013  531.16560
3   2019-08-01 03:00:00       0.0  28.700806  499.77240
4   2019-08-01 04:00:00       0.0  28.653845  465.63425
..                  ...       ...        ...        ...
693 2019-08-31 19:00:00       0.0  28.352089  462.41962
694 2019-08-31 20:00:00       0.0  28.626904  436.77590
695 2019-08-31 21:00:00       0.0  28.583805  434.29420
696 2019-08-31 22:00:00       0.0  28.175760  443.68936
697 2019-08-31 23:00:00       0.0  27.949297  445.04990

[698 rows x 4 columns]


In [14]:
pred_all_without_datetime = pred_all_final.drop(columns=['datetime'])

pred_all_without_datetime.to_csv('pred_all.csv', index=False)